 https://riscv.org/wp-content/uploads/2024/12/15.20-15.55-18.05.06.VEXT-bcn-v1.pdf

https://github.com/NationalSecurityAgency/ghidra/discussions/5744

https://github.com/NationalSecurityAgency/ghidra/blob/master/Ghidra/Processors/RISCV/data/languages/riscv.rvv.sinc
They possibly just lift as opaque though

In [1]:
%%file /tmp/riscv.s
.option arch, +v
.text
.globl _start
_start:
    vsetvli t0, zero, e32, m1, ta, ma
    vle32.v v1, (a0)
    vle32.v v2, (a1)
    vadd.vv v3, v1, v2
    vse32.v v3, (a2)


Writing /tmp/riscv.s


In [2]:
!riscv64-unknown-elf-as -march=rv64gcv /tmp/riscv.s -o /tmp/riscv.o

In [3]:
!riscv64-unknown-elf-objdump -d /tmp/riscv.o


/tmp/riscv.o:     file format elf64-littleriscv


Disassembly of section .text:

0000000000000000 <_start>:
   0:	0d0072d7          	vsetvli	t0,zero,e32,m1,ta,ma
   4:	02056087          	vle32.v	v1,(a0)
   8:	0205e107          	vle32.v	v2,(a1)
   c:	021101d7          	vadd.vv	v3,v1,v2
  10:	020661a7          	vse32.v	v3,(a2)


In [5]:
!riscv64-unknown-elf-objcopy -O binary -j .text /tmp/riscv.o /tmp/riscv.bin

In [6]:
from pathlib import Path

import pypcode

code = Path("/tmp/riscv.bin").read_bytes()
print(code.hex())

ctx = pypcode.Context("RISCV:LE:64:RV64G")
dx = ctx.disassemble(code, 0)
for ins in dx.instructions:
    print(f"{ins.addr.offset:#x}/{ins.length}: {ins.mnem} {ins.body}")

print("--- pcode ---")
for ins in dx.instructions:
    chunk = code[ins.addr.offset : ins.addr.offset + ins.length]
    try:
        tx = ctx.translate(chunk, ins.addr.offset)
        print(f"{ins.addr.offset:#x}: {len(tx.ops)} ops")
        for op in tx.ops:
            print(op)
    except Exception as e:
        print(f"{ins.addr.offset:#x}: {type(e).__name__}: {e}")


d772000d8760050207e10502d7011102a7610602
0x0/4: vsetvli t0, zero, 0xd0
0x4/4: vle32.v v1, (a0), 0x1
0x8/4: vle32.v v2, (a1), 0x1
0xc/4: vadd.vv v3, v1, v2, 0x1
0x10/4: vse32.v v3, (a2), 0x1
--- pcode ---
0x0: UnimplError: Instruction not implemented in pcode:
 0x00000000: vsetvli  t0, zero, 0xd0
0x4: UnimplError: Instruction not implemented in pcode:
 0x00000004: vle32.v  v1, (a0), 0x1
0x8: UnimplError: Instruction not implemented in pcode:
 0x00000008: vle32.v  v2, (a1), 0x1
0xc: UnimplError: Instruction not implemented in pcode:
 0x0000000c: vadd.vv  v3, v1, v2, 0x1
0x10: UnimplError: Instruction not implemented in pcode:
 0x00000010: vse32.v  v3, (a2), 0x1
